# 🌧️ Análisis de Precipitación – IDEAM
## Valle del Cauca · Estaciones Automáticas

---

### 📌 Contexto

Este notebook analiza los datos de **precipitación** del **IDEAM** (Instituto de Hidrología, Meteorología y Estudios Ambientales) provenientes de estaciones automáticas ubicadas en el Valle del Cauca.

Los datos originales son mediciones cada **10 minutos** (165M+ registros nacionales). Para nuestro análisis, ya fueron agregados a nivel **mensual** y **anual** por estación y municipio.

### 🎯 Objetivo del análisis
1. Mapear la cobertura de estaciones meteorológicas en el Valle del Cauca
2. Analizar patrones de precipitación mensual y anual
3. Identificar estacionalidad y tendencias climáticas
4. Evaluar la utilidad como **predictor climático** para el modelo RFRK

> **Fuente**: [datos.gov.co – Precipitación IDEAM](https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Precipitaci-n/s54a-sgyg)

---
## 1️⃣ Configuración y carga de datos

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import os

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.dpi': 100,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'font.family': 'sans-serif',
})

REPORT_DIR = os.path.join('../data', '../reports')
MESES_ES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

print('✅ Librerías cargadas')

✅ Librerías cargadas


In [4]:
# Cargar datasets procesados
DATA_DIR = os.path.join('data', 'processed')

df_mensual_est = pd.read_csv(os.path.join(DATA_DIR, 'ideam_precipitacion_mensual_estacion.csv'))
df_mensual_muni = pd.read_csv(os.path.join(DATA_DIR, 'ideam_precipitacion_mensual_municipio.csv'))
df_anual = pd.read_csv(os.path.join(DATA_DIR, 'ideam_precipitacion_anual_municipio.csv'))
df_catalogo = pd.read_csv(os.path.join(DATA_DIR, 'ideam_catalogo_estaciones_valle.csv'))

print(f'📊 Mensual por estación:  {df_mensual_est.shape[0]:>8,} registros')
print(f'📊 Mensual por municipio: {df_mensual_muni.shape[0]:>8,} registros')
print(f'📊 Anual por municipio:   {df_anual.shape[0]:>8,} registros')
print(f'📊 Catálogo estaciones:   {df_catalogo.shape[0]:>8,} estaciones')
print(f'\n🗺️ Municipios con datos: {df_mensual_muni["municipio"].nunique()}')
print(f'📅 Rango temporal: {df_anual["anio"].min()} – {df_anual["anio"].max()}')

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\ideam_precipitacion_mensual_estacion.csv'

---
## 2️⃣ Mapa de estaciones meteorológicas

In [ ]:
# Mapa interactivo de estaciones
fig = px.scatter_mapbox(
    df_catalogo,
    lat='latitud', lon='longitud',
    color='municipio',
    size='total_mediciones',
    hover_name='nombreestacion',
    hover_data=['codigoestacion', 'municipio', 'total_mediciones'],
    title='Estaciones Meteorológicas IDEAM – Valle del Cauca',
    mapbox_style='carto-positron',
    zoom=7,
    height=600,
    size_max=20,
)
fig.update_layout(
    font=dict(size=13),
    title_font_size=18,
    margin=dict(l=0, r=0, t=50, b=0)
)
fig.show()

In [ ]:
# Tabla de estaciones
df_catalogo['fecha_inicio'] = pd.to_datetime(df_catalogo['fecha_inicio']).dt.strftime('%Y-%m-%d')
df_catalogo['fecha_fin'] = pd.to_datetime(df_catalogo['fecha_fin']).dt.strftime('%Y-%m-%d')
print(f'📋 Catálogo de {len(df_catalogo)} estaciones meteorológicas en el Valle del Cauca:\n')
df_catalogo[['nombreestacion', 'municipio', 'latitud', 'longitud', 'fecha_inicio', 'fecha_fin', 'total_mediciones']].sort_values('total_mediciones', ascending=False)

---
## 3️⃣ Precipitación anual por municipio

In [ ]:
# Precipitación anual promedio por municipio
precip_promedio = df_anual.groupby('municipio')['precipitacion_anual_mm'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, max(8, len(precip_promedio) * 0.5)))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.8, len(precip_promedio)))
bars = ax.barh(precip_promedio.index[::-1], precip_promedio.values[::-1], color=colors)
ax.set_xlabel('Precipitación anual promedio (mm)')
ax.set_title('Precipitación Anual Promedio por Municipio\n(Valle del Cauca)')

for bar, val in zip(bars, precip_promedio.values[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f} mm', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_precipitacion_anual_municipio.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evolución de la precipitación anual (todos los municipios)
fig = px.line(df_anual, x='anio', y='precipitacion_anual_mm', color='municipio',
              markers=True,
              title='Evolución de la Precipitación Anual por Municipio',
              labels={'precipitacion_anual_mm': 'Precipitación Anual (mm)',
                      'anio': 'Año', 'municipio': 'Municipio'},
              template='plotly_white')
fig.update_traces(line=dict(width=2), marker=dict(size=6))
fig.update_layout(font=dict(size=13), title_font_size=16, height=500)
fig.show()

---
## 4️⃣ Estacionalidad de la precipitación (patrón mensual)

In [ ]:
# Patrón mensual promedio (todos los municipios combinados)
patron_mensual = df_mensual_muni.groupby('mes')['precipitacion_promedio_mm'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
colors_month = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12))
bars = ax.bar(patron_mensual['mes'], patron_mensual['precipitacion_promedio_mm'],
              color=colors_month, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES_ES)
ax.set_xlabel('Mes')
ax.set_ylabel('Precipitación promedio (mm)')
ax.set_title('Estacionalidad de la Precipitación en el Valle del Cauca\n(Promedio mensual de todas las estaciones)')

for bar, val in zip(bars, patron_mensual['precipitacion_promedio_mm']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}', ha='center', fontweight='bold', fontsize=10)

ax.axhline(y=patron_mensual['precipitacion_promedio_mm'].mean(), color='red',
           linestyle='--', alpha=0.5, label=f'Promedio: {patron_mensual["precipitacion_promedio_mm"].mean():.0f} mm')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_estacionalidad_precipitacion.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📝 Se observa el patrón bimodal típico del Valle del Cauca (dos temporadas de lluvia)')

In [ ]:
# Heatmap: Precipitación mensual por municipio
pivot_muni_mes = df_mensual_muni.groupby(['municipio', 'mes'])['precipitacion_promedio_mm'].mean().unstack(fill_value=0)
pivot_muni_mes.columns = MESES_ES

fig, ax = plt.subplots(figsize=(14, max(8, len(pivot_muni_mes) * 0.5)))
sns.heatmap(pivot_muni_mes, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Precipitación promedio (mm)'})
ax.set_title('Precipitación Mensual Promedio por Municipio\n(Valle del Cauca – mm)')
ax.set_ylabel('Municipio')
ax.set_xlabel('Mes')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_heatmap_precipitacion_mensual.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5️⃣ Análisis por estación meteorológica

In [ ]:
# Precipitación acumulada por estación y año
est_anual = df_mensual_est.groupby(['nombreestacion', 'municipio', 'anio']).agg(
    precipitacion_anual=('precipitacion_acumulada_mm', 'sum')
).reset_index()

# Top 10 estaciones con más precipitación promedio anual
top_est = est_anual.groupby('nombreestacion')['precipitacion_anual'].mean().nlargest(10).index
est_top = est_anual[est_anual['nombreestacion'].isin(top_est)]

fig = px.line(est_top, x='anio', y='precipitacion_anual', color='nombreestacion',
              markers=True,
              title='Top 10 Estaciones con Mayor Precipitación Anual',
              labels={'precipitacion_anual': 'Precipitación Anual (mm)',
                      'anio': 'Año', 'nombreestacion': 'Estación'},
              template='plotly_white')
fig.update_layout(font=dict(size=13), title_font_size=16, height=500)
fig.show()

In [ ]:
# Distribución de la precipitación acumulada mensual por estación
fig, ax = plt.subplots(figsize=(14, 7))
df_plot = df_mensual_est[df_mensual_est['precipitacion_acumulada_mm'] > 0]
sns.boxplot(data=df_plot, x='mes', y='precipitacion_acumulada_mm', 
            palette='YlGnBu', ax=ax, showfliers=False)
ax.set_xticklabels(MESES_ES)
ax.set_xlabel('Mes')
ax.set_ylabel('Precipitación acumulada mensual (mm)')
ax.set_title('Distribución de la Precipitación Mensual por Estación\n(Valle del Cauca)')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_boxplot_precipitacion_mensual.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6️⃣ Días con lluvia y frecuencia

In [ ]:
# Días con lluvia promedio por mes
dias_lluvia = df_mensual_muni.groupby('mes')['dias_con_lluvia_promedio'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Días con lluvia
axes[0].bar(dias_lluvia['mes'], dias_lluvia['dias_con_lluvia_promedio'],
           color=plt.cm.Blues(np.linspace(0.3, 0.9, 12)), edgecolor='white')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(MESES_ES)
axes[0].set_ylabel('Registros con lluvia (promedio)')
axes[0].set_title('Frecuencia de Registros con Lluvia por Mes')

# Intensidad vs frecuencia (scatter)
mensual_agg = df_mensual_muni.groupby('mes').agg(
    precipitacion=('precipitacion_promedio_mm', 'mean'),
    dias_lluvia=('dias_con_lluvia_promedio', 'mean')
).reset_index()

axes[1].scatter(mensual_agg['dias_lluvia'], mensual_agg['precipitacion'],
               s=200, c=range(12), cmap='RdYlBu_r', edgecolors='black', zorder=5)
for _, row in mensual_agg.iterrows():
    axes[1].annotate(MESES_ES[int(row['mes'])-1],
                    (row['dias_lluvia'], row['precipitacion']),
                    textcoords='offset points', xytext=(8, 5), fontweight='bold')
axes[1].set_xlabel('Registros con lluvia (promedio)')
axes[1].set_ylabel('Precipitación promedio (mm)')
axes[1].set_title('Relación: Frecuencia de Lluvia vs Cantidad')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_dias_lluvia.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7️⃣ Variabilidad interanual

In [ ]:
# Variabilidad de la precipitación anual
precip_general = df_anual.groupby('anio')['precipitacion_anual_mm'].agg(['mean', 'std', 'min', 'max']).reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
ax.fill_between(precip_general['anio'], precip_general['min'], precip_general['max'],
               alpha=0.15, color='steelblue', label='Rango min-max')
ax.fill_between(precip_general['anio'],
               precip_general['mean'] - precip_general['std'],
               precip_general['mean'] + precip_general['std'],
               alpha=0.3, color='steelblue', label='± 1 desv. estándar')
ax.plot(precip_general['anio'], precip_general['mean'], 'o-', color='navy',
       linewidth=2.5, markersize=8, label='Promedio')

ax.set_xlabel('Año')
ax.set_ylabel('Precipitación Anual (mm)')
ax.set_title('Variabilidad Interanual de la Precipitación\n(Valle del Cauca – Todos los municipios)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_variabilidad_interanual.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Precipitación anual por municipio y año
pivot_anual = df_anual.pivot_table(index='municipio', columns='anio',
                                    values='precipitacion_anual_mm', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, max(8, len(pivot_anual) * 0.5)))
sns.heatmap(pivot_anual, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Precipitación anual (mm)'})
ax.set_title('Precipitación Anual por Municipio y Año\n(Valle del Cauca – mm)')
ax.set_ylabel('Municipio')
ax.set_xlabel('Año')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_heatmap_precipitacion_anual.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 8️⃣ Cobertura de datos y calidad

In [ ]:
# Cobertura temporal por estación (¿desde cuándo y hasta cuándo tiene datos cada estación?)
df_catalogo_dt = pd.read_csv(os.path.join(DATA_DIR, 'ideam_catalogo_estaciones_valle.csv'))
df_catalogo_dt['fecha_inicio'] = pd.to_datetime(df_catalogo_dt['fecha_inicio'])
df_catalogo_dt['fecha_fin'] = pd.to_datetime(df_catalogo_dt['fecha_fin'])
df_catalogo_dt['anios_datos'] = (df_catalogo_dt['fecha_fin'] - df_catalogo_dt['fecha_inicio']).dt.days / 365.25

fig, ax = plt.subplots(figsize=(14, max(8, len(df_catalogo_dt) * 0.4)))
y_pos = range(len(df_catalogo_dt))

for i, (_, row) in enumerate(df_catalogo_dt.sort_values('fecha_inicio').iterrows()):
    ax.barh(i, (row['fecha_fin'] - row['fecha_inicio']).days,
           left=row['fecha_inicio'].toordinal(),
           height=0.7, color=plt.cm.tab20(i % 20), alpha=0.8)

ax.set_yticks(range(len(df_catalogo_dt)))
ax.set_yticklabels(df_catalogo_dt.sort_values('fecha_inicio')['nombreestacion'].values, fontsize=8)
ax.set_xlabel('Periodo')
ax.set_title('Cobertura Temporal de cada Estación Meteorológica')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: pd.Timestamp.fromordinal(int(x)).strftime('%Y')))

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_cobertura_temporal_estaciones.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Meses con datos por municipio y año (para detectar gaps)
cobertura = df_anual[['municipio', 'anio', 'meses_con_datos']].copy()
pivot_cob = cobertura.pivot_table(index='municipio', columns='anio', values='meses_con_datos')

fig, ax = plt.subplots(figsize=(14, max(8, len(pivot_cob) * 0.5)))
sns.heatmap(pivot_cob, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=12,
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Meses con datos (de 12)'})
ax.set_title('Completitud: Meses con Datos por Municipio y Año\n(12 = año completo, <12 = datos incompletos)')
ax.set_ylabel('Municipio')
ax.set_xlabel('Año')

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'fig_completitud_temporal.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9️⃣ Cruce potencial con datos EVA

In [ ]:
# Verificar qué municipios del EVA tienen datos de precipitación
try:
    df_eva = pd.read_csv(os.path.join(DATA_DIR, 'eva_valle_cultivos_interes.csv'))
    
    munis_eva = set(df_eva['municipio'].str.upper().unique())
    munis_ideam = set(df_mensual_muni['municipio'].str.upper().unique())
    
    en_ambos = munis_eva & munis_ideam
    solo_eva = munis_eva - munis_ideam
    solo_ideam = munis_ideam - munis_eva
    
    print('🔗 CRUCE EVA ↔ IDEAM (por municipio):')
    print(f'\n   Municipios con datos en AMBOS datasets: {len(en_ambos)}')
    for m in sorted(en_ambos):
        print(f'     ✅ {m}')
    
    print(f'\n   Municipios SOLO en EVA (sin datos de lluvia): {len(solo_eva)}')
    for m in sorted(solo_eva):
        print(f'     ⚠️ {m}')
    
    print(f'\n   Municipios SOLO en IDEAM (sin datos de cultivos): {len(solo_ideam)}')
    for m in sorted(solo_ideam):
        print(f'     ℹ️ {m}')
    
    # Visualizar
    fig, ax = plt.subplots(figsize=(8, 8))
    from matplotlib_venn import venn2
    try:
        venn2([munis_eva, munis_ideam], set_labels=('EVA\n(cultivos)', 'IDEAM\n(precipitación)'),
              ax=ax, set_colors=('#F2C94C', '#3498DB'), alpha=0.6)
        ax.set_title('Cruce de Municipios: EVA vs IDEAM')
    except ImportError:
        sizes = [len(solo_eva), len(en_ambos), len(solo_ideam)]
        labels = [f'Solo EVA\n({sizes[0]})', f'Ambos\n({sizes[1]})', f'Solo IDEAM\n({sizes[2]})']
        ax.bar(labels, sizes, color=['#F2C94C', '#27AE60', '#3498DB'])
        ax.set_title('Cruce de Municipios: EVA vs IDEAM')
        ax.set_ylabel('Número de municipios')
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, 'fig_cruce_eva_ideam.png'), dpi=150, bbox_inches='tight')
    plt.show()

except FileNotFoundError:
    print('⚠️ No se encontró el dataset EVA procesado. Ejecuta primero el notebook 01_analisis_eva.ipynb')

---
## 🔟 Resumen ejecutivo y conclusiones

### ✅ Hallazgos principales

| Aspecto | Resultado |
|---------|----------|
| **Tipo de datos** | Precipitación en mm (mediciones automáticas cada 10 min) |
| **Resolución procesada** | Mensual y anual por estación y municipio |
| **Estaciones en Valle del Cauca** | Múltiples estaciones con coordenadas GPS |
| **Patrón climático** | Bimodal (dos temporadas de lluvia al año) |
| **Dato importante** | Datos crudos, NO validados oficialmente por IDEAM |

### 🎯 Evaluación para el modelo RFRK

| Criterio | Evaluación |
|----------|------------|
| ¿Es útil como predictor? | ✅ **SÍ** – Precipitación es clave para rendimientos agrícolas |
| ¿Tiene coordenadas GPS? | ✅ **SÍ** – Latitud y longitud de cada estación (útil para Kriging) |
| ¿Se puede cruzar con EVA? | ✅ **SÍ** – Por municipio y año |
| ¿Cobertura temporal? | ⚠️ Variable por estación (gaps en algunos períodos) |
| ¿Datos validados? | ⚠️ NO – Son datos crudos con control de calidad básico |

### 🚀 Siguiente paso
Descargar datos topográficos (**DEM SRTM**) para completar el triángulo de predictores: clima + topografía + rendimientos.

In [ ]:
# Resumen final
print('=' * 60)
print('RESUMEN DEL DATASET PRECIPITACIÓN IDEAM – VALLE DEL CAUCA')
print('=' * 60)
print(f'Fuente: IDEAM – datos.gov.co (ID: s54a-sgyg)')
print(f'Estaciones: {len(df_catalogo)}')
print(f'Municipios con datos: {df_mensual_muni["municipio"].nunique()}')
print(f'Rango temporal: {df_anual["anio"].min()} – {df_anual["anio"].max()}')
print(f'\nArchivos procesados:')
print(f'  - Mensual/estación:  {len(df_mensual_est):,} registros')
print(f'  - Mensual/municipio: {len(df_mensual_muni):,} registros')
print(f'  - Anual/municipio:   {len(df_anual):,} registros')
print(f'  - Catálogo:          {len(df_catalogo)} estaciones')
print(f'\n✅ DATASET VALIDADO Y APTO COMO PREDICTOR CLIMÁTICO')